This is a notebook for computing deep foundation axial geotechnical capacity. <br>
Author: Zhiyan Jiang [(linkedin.com/in/zhiyanjiang)](http://www.linkedin.com/in/zhiyanjiang)

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
#import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

In [3]:
from deep_foundation_bearing_capacity.cross_sections.cross_sections import CircularSection, CrossSection
from deep_foundation_bearing_capacity.factor_of_safety.factor_of_safety import FactorOfSafetyDeepFoundation
from deep_foundation_bearing_capacity.foundation.deep_foundation import DeepFoundation
from deep_foundation_bearing_capacity.foundation.foundation_material import FoundationConcrete
from deep_foundation_bearing_capacity.geomaterials.layer import Layer
from deep_foundation_bearing_capacity.geomaterials.rock import Rock
from deep_foundation_bearing_capacity.geomaterials.soil import Soil
from deep_foundation_bearing_capacity.segments.segments import Segment
from deep_foundation_bearing_capacity.segments.unit_resistance import SideResistance, SoilEndResistance

In [4]:
def _find_project_root(marker = "pyproject.toml"):
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / marker).exists():
            return p
        else:
            p = p.parent
    raise FileNotFoundError(f"No {marker} found")

PROJECT_ROOT = _find_project_root()
#print(f"Project root is: {PROJECT_ROOT}")

In [5]:
# Read rock parameters from YAML
with open(PROJECT_ROOT/"data/rock_params.yaml") as f:
    rocks_dict = yaml.safe_load(f)

rock_list = []
for rock_dict in rocks_dict:
    rock = Rock.from_dict(rock_dict)
    rock_list.append(rock)

    rock.display_properties()
    print()
    

rock_index is: 0
unit_weight is: 150.0
friction_angle is: 30.0
qu is: 10000.0
rqd is: 50.0
rock_type is: A
rock_quality is: Good
rock_type_advanced is: None
joint is: closed

rock_index is: 1
unit_weight is: 150.0
friction_angle is: 27.0
qu is: 10000.0
rqd is: 0.5
rock_type is: A
rock_quality is: Good
rock_type_advanced is: None
joint is: closed

rock_index is: 2
unit_weight is: 150.0
friction_angle is: 30.0
qu is: 10000.0
rqd is: 0.8
rock_type is: A
rock_quality is: Good
rock_type_advanced is: None
joint is: closed



In [6]:
# Read layer parameters from YAML

with open(PROJECT_ROOT/"data/layer_params1.yaml") as f:
    layers_dict = yaml.safe_load(f)


layer_list = []
for layer_dict in layers_dict:

    layer = Layer.from_dict(layer_dict)
    layer_list.append(layer)
    

# Manually assign geomaterial
layer_list[0].geomaterial = rock_list[0]
layer_list[1].geomaterial = rock_list[1]
layer_list[2].geomaterial = rock_list[2]

for layer in layer_list:
    layer.display_properties(display_geomaterial=True)
    print()


layer_index is: 0
ground_water_depth is: 15.0
top_depth is: 0
thickness is: 10
rock_index is: 0
unit_weight is: 150.0
friction_angle is: 30.0
qu is: 10000.0
rqd is: 50.0
rock_type is: A
rock_quality is: Good
rock_type_advanced is: None
joint is: closed

layer_index is: 1
ground_water_depth is: 15.0
top_depth is: 10
thickness is: 5
rock_index is: 1
unit_weight is: 150.0
friction_angle is: 27.0
qu is: 10000.0
rqd is: 0.5
rock_type is: A
rock_quality is: Good
rock_type_advanced is: None
joint is: closed

layer_index is: 2
ground_water_depth is: 15.0
top_depth is: 15
thickness is: 10
rock_index is: 2
unit_weight is: 150.0
friction_angle is: 30.0
qu is: 10000.0
rqd is: 0.8
rock_type is: A
rock_quality is: Good
rock_type_advanced is: None
joint is: closed



In [ ]:
np.interp(50.0, [100.0, 70.0, 50.0, 30.0, 20.0], [1.00, 0.85, 0.60, 0.50, 0.45])

NameError: name 'numpy' is not defined

In [10]:
# test SideResistance class for cohesionless IGM layer
side_resistance_obj_1 = SideResistance.for_material(layer_list[0])
print(f"First layer alpha for cohesive IGM is: {side_resistance_obj_1._calculate_alpha():.2f}")
print(f"First layer phi is: {side_resistance_obj_1._calculate_phi()}")
print(f"First layer ko is: {side_resistance_obj_1._calculate_ko(750)}")
print(f"First layer ko is: {side_resistance_obj_1.side_resistance_unit(750)}")

# test SideResistance class for cohesive layer
side_resistance_obj_2 = SideResistance.for_material(layer_list[1])
print(f"Second layer alpha is: {side_resistance_obj_2._calculate_alpha():.2f}")

First layer alpha for cohesive IGM is: 0.20
[1.0, 0.85, 0.6, 0.5, 0.45]
50.0
First layer phi is: 0.45


AttributeError: 'RockSideResistance' object has no attribute '_calculate_ko'

In [ ]:
list(range(1,8))

In [ ]:
# test EndResistance class for cohesionless layer
end_resistance_obj_1 = SoilEndResistance(layer_list[0])
print(f"Top layer unit end resistance is: {end_resistance_obj_1.end_resistance_unit():.2f}")

In [ ]:
# test EndResistance class for cohesionless layer
end_resistance_obj_2 = SoilEndResistance(layer_list[1])
print(f"Second layer unit end resistance is: {end_resistance_obj_2.end_resistance_unit():.2f}")

In [ ]:
# test CrossSection class
cross_section_diameter = 1.5
print(f"Cross section diameter is: {cross_section_diameter}")
cross_section_obj = CircularSection(1.5)
print(f"Cross section area is: {cross_section_obj.cross_section_area:.2f}")
print(f"Cross section perimeter is: {cross_section_obj.perimeter:.2f}")

In [ ]:
# test Segment class
segment_obj_list = []
foundation_concrete_obj = FoundationConcrete()

for layer in layer_list:
    segment_obj = Segment(cross_section_obj, layer, foundation_concrete_obj)
    segment_obj_list.append(segment_obj)



In [ ]:
print(f"Top segment side surface area is: {segment_obj_list[0].side_surface_area:.2f}")

In [ ]:
# test DeepFoundation class
top_depth = 0

deep_foundation_obj = DeepFoundation(segment_obj_list, top_depth,  resistance_corrections = None)

# test DeepFoundation method for self weight
print(f"Concrete foundation self weight is: {deep_foundation_obj.effective_weight:.0f} lb.")

In [ ]:
# test DeepFoundation method for calculating effective stress
print(f"Deep foundation effective stresses are: {deep_foundation_obj.calculate_segment_effective_stresses}")

In [ ]:
# test FactorOfSafety class
fs = 3.0
print(f"Factor of safety is prescribed as: {fs}")
fs_obj = FactorOfSafetyDeepFoundation(3, 3)

In [ ]:
# test DeepFoundation side resistance
deep_foundation_obj.calculate_segment_side_resistances(uplift = False)

In [ ]:
# test DeepFoundation side resistance
deep_foundation_obj.calculate_segment_side_resistances(uplift = True)

In [ ]:
# test DeepFoundation accumulative side resistance 
for segment_side_resistance_accumulative, segment_end_resistance in zip(
deep_foundation_obj.calculate_segment_side_resistances_accumulative(),
deep_foundation_obj.calculate_segment_end_resistances()
):
    print(f"{segment_side_resistance_accumulative:.0f}, {segment_end_resistance:.0f}")

In [ ]:
# test DeepFoundation accumulative compression resistance
print("Accumulative compression resistance is:")
for temp in deep_foundation_obj.calculate_compression_resistances_accumulative():
    print(f"{temp:.0f} lb.")

In [ ]:
# test DeepFoundation calculate_segment_weights_accumulative
print("Accumulative self weight is: ")
deep_foundation_obj.calculate_segment_weights_effective_accumulative()

In [ ]:
# test DeepFoundation accumulative uplift resistance
print("Accumulative compression resistance is:")
for temp in deep_foundation_obj.calculate_uplift_resistances_accumulative():
    print(f"{temp:.0f} lb.")

In [ ]:
# Test plotting compression accumulative resistancedeep_foundation_obj.visualize_resistances_accumulative(target = "compression", style = "piecewise")

In [ ]:
deep_foundation_obj.visualize_resistances_accumulative(target = "compression", style = "mid")

In [ ]:
# Test plotting uplift accumulative resistance
deep_foundation_obj.visualize_resistances_accumulative(target = "uplift", style = "piecewise")


In [ ]:
deep_foundation_obj.visualize_resistances_accumulative(target = "uplift", style = "mid")

In [ ]:
# Test structural compression capacity
print("Each segment's structural capacity in compression is: ")
deep_foundation_obj.segments_structural_capacity_compression()

In [ ]:
# Test side resistance correction
#deep_foundation_obj = DeepFoundation([segment_obj_1, segment_obj_2], top_depth, resistance_correction=True)
#print(f"With depth correction, second segment side resistance is: {deep_foundation_obj.segments[1].side_resistance}")